# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Abdrakhmanova Adel

**Student ID:** S23067852

**Group:** CSS4007-ENG-10

**Repository:** https://github.com/dellya4/css4007-hw3-dellya4

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>Using ChatGPT for understanding, explanations and debugging

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens                                                                                          | \|d\| |
|-----|-------------------------------------------------------------------------------------------------|------:|
| D1  | large, language, models, can, generate, text, and, answer, questions                            |     9 |
| D2  | retrieval, augmented, generation, retrieves, relevant, document, before, generating, an, answer |    10 |
| D3  | embedding, represent, text, as, numerical, vectors                                              |     6 |

**avgdl = (9+10+6)/3 = 8.333**

**Query terms: how, does, rag, retrieve, information**

**TF, DF and IDF of each query term:**

| Query term  | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|-------------|---------:|---------:|---------:|----------:|-------:|
| how         |        0 |        0 |        0 |         0 | 1.9459 |
| does        |        0 |        0 |        0 |         0 | 1.9459 |
| RAG         |        0 |        0 |        0 |         0 | 1.9459 |
| retrieve    |        0 |        0 |        0 |         0 | 1.9459 |
| information |        0 |        0 |        0 |         0 | 1.9459 |

n(t)=0, N=3 - for every query term
idf(t) = ln ((N-n(t)+0.5)/(n(t)+0.5) =
ln ((3-0+0.5)/(0+0.5)) =
ln (3.5/0.5) =
ln 7 = 1.9459

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

>For D1:
>BM25(D1) = 0 + 0 + 0 + 0 + 0 = 0

>For D2:
>BM25(D2) = 0 + 0 + 0 + 0 + 0 = 0

>For D3:
>BM25(D3) = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|-----------:|-----:|
| D1 |     0.0000 |    1 |
| D2 |     0.0000 |    1 |
| D3 |     0.0000 |    1 |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens                                                                             | \|d\| |
|-----|------------------------------------------------------------------------------------|------:|
| D1  | large, language, models, can, gener, text, and, answer, questions                  |       |
| D2  | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer |       |
| D3  | embeddings, represent, text, as, numerical, vectors                                |       |

**avgdl = (9+10+6)/3 = 8.333**

**Stemmed query terms: how, doe, rag, retriev, inform **

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|------------|---------:|---------:|---------:|----------:|-------:|
| how        |        0 |        0 |        0 |         0 | 1.9459 |
| doe        |        0 |        0 |        0 |         0 | 1.9459 |
| rag        |        0 |        0 |        0 |         0 | 1.9459 |
| retriev    |        0 |        2 |        0 |         1 | 0.5108 |
| inform     |        0 |        0 |        0 |         0 | 1.9459 |

n(t)=0, N=3 - for every query term (except for D2)
idf(t) = ln ((N-n(t)+0.5)/(n(t)+0.5) =
ln ((3-0+0.5)/(0+0.5)) =
ln (3.5/0.5) =
ln 7 = 1.9459

for D2: n(t)=1, N=3
idf(t) = ln ((N-n(t)+0.5)/(n(t)+0.5) =
ln ((3-1+0.5)/(1+0.5)) =
ln (2.5/1.5) =
ln 1.6667 = 0.5108

**BM25 of each document**, worked:

>For D1:
>BM25(D1) = 0 + 0 + 0 + 0 + 0 = 0

>For D2:
>BM25(D2) = 0.5108*(2*(1.5+1)/(2+1.5(1-0.75+0.75*(10/8.3333))=
>= 0.5108*(5/3.725)
>= 0.5108*1.3423=0.6857

>For D3:
>BM25(D3) = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|-----------:|-----:|
| D1 |          0 |    2 |
| D2 |     0.6857 |    1 |
| D3 |          0 |    2 |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> (a) Each document have a first place because don't have similarity
> (b) D2 document retrieve first, because has 2 similarity moments (retriev)

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ |    ‖D‖ | Cosine |
|---|----------------------|--------:|-------:|-------:|
| Query · D1 | 0.78                 |  0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 1.08                 |  0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.34                 |  0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|------------------:|-----:|
| D1 |            0.9956 |    2 |
| D2 |            0.9977 |    1 |
| D3 |            0.4244 |    3 |

**Which document is most similar to the query?**

> D2 was most similar to query because have a most big cosine similarity

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> If we compare with stem's result - yes, it compares. With the first result our output don't add up because in (a) result we don't have the winner, all strings were on the same level

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

>BM25 without stemming exactly checks the tokens, they must match completely. But meaning in query and document can be similar. Embedding vector check document and query for meaning, and can find answer better.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [2]:
import math
import numpy as np

N = 3
k1 = 1.5
b = 0.75

doc_lengths = [9, 10, 6]

avgdl = sum(doc_lengths) / len(doc_lengths)

print("Average document length:", avgdl)

n = 1

idf = math.log(
    (N - n + 0.5) /
    (n + 0.5)
)

print("IDF(retriev):", idf)

f = 2
dl = 10

score_d2 = (
    idf
    * f
    * (k1 + 1)
    / (
        f
        + k1 * (
            1 - b
            + b * dl / avgdl
        )
    )
)

print("BM25 D2:", score_d2)

query = np.array([0.8, 0.2, 0.5])

documents = {
    "D1": np.array([0.7, 0.1, 0.4]),
    "D2": np.array([0.9, 0.3, 0.6]),
    "D3": np.array([0.1, 0.8, 0.2])
}

query_magnitude = np.linalg.norm(query)

for name, document in documents.items():

    dot_product = np.dot(query, document)

    document_magnitude = np.linalg.norm(document)

    cosine = (
        dot_product /
        (query_magnitude * document_magnitude)
    )

    print(name)
    print("Dot product:", dot_product)
    print("Query magnitude:", query_magnitude)
    print("Document magnitude:", document_magnitude)
    print("Cosine:", cosine)
    print()

Average document length: 8.333333333333334
IDF(retriev): 0.5108256237659907
BM25 D2: 0.6856719782093835
D1
Dot product: 0.78
Query magnitude: 0.9643650760992956
Document magnitude: 0.812403840463596
Cosine: 0.9955914553877285

D2
Dot product: 1.08
Query magnitude: 0.9643650760992956
Document magnitude: 1.1224972160321824
Cosine: 0.9976931918526478

D3
Dot product: 0.3400000000000001
Query magnitude: 0.9643650760992956
Document magnitude: 0.8306623862918076
Cosine: 0.42443666888811166



---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [3]:
import json
import re
import numpy as np

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\asely\PycharmProjects\css4007-hw3-dellya4\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
corpus = []

with open("data/corpus.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        corpus.append(json.loads(line))

with open("data/queries.json", "r", encoding="utf-8") as file:
    queries = json.load(file)

print("Number of corpus chunks:", len(corpus))
print("Number of fixed queries:", len(queries["queries"]))

Number of corpus chunks: 74
Number of fixed queries: 8


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>I lowercase the text and use the regex `\w+` to extract word tokens.
>I don't use stemming and I do not remove stop words.

In [5]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

In [6]:
tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in corpus
]

print("Example tokens:")
print(tokenized_corpus[0][:20])

Example tokens:
['kazakhstan', 'officially', 'the', 'republic', 'of', 'kazakhstan', 'is', 'a', 'landlocked', 'country', 'situated', 'primarily', 'in', 'central', 'asia', 'with', 'a', 'portion', 'of', 'its']


In [7]:
bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

print("BM25 index created.")

BM25 index created.


In [8]:
def bm25_retrieve(query, top_k=3):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "id": corpus[index]["id"],
            "title": corpus[index]["title"],
            "text": corpus[index]["text"],
            "score": float(scores[index])
        })

    return results

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

print("HF token found:", hf_token is not None)

HF token found: True


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=hf_token
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 613.37it/s]


Model loaded successfully!


In [16]:
document_texts = [
    chunk["text"]
    for chunk in corpus
]

document_embeddings = embedding_model.encode_document(
    document_texts,
    show_progress_bar=True
)

print("Document embeddings shape:", document_embeddings.shape)

Batches: 100%|██████████| 3/3 [00:39<00:00, 13.17s/it]

Document embeddings shape: (74, 768)


In [11]:
def embedding_retrieve(query, top_k=3):
    query_embedding = embedding_model.encode_query(
        query
    )

    scores = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "id": corpus[index]["id"],
            "title": corpus[index]["title"],
            "text": corpus[index]["text"],
            "score": float(scores[index])
        })

    return results

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id   | Query                                                                          | `relevant` | Expected winner, and why                                                                     |
|------|--------------------------------------------------------------------------------|------------|----------------------------------------------------------------------------------------------|
| Q-09 | Who was Nur-Sultan named after in 2019?                                        | KZ-05      | BM25, because in question I have a lot of words which we have in KZ-05 document.             |
| Q-10 | Which language's writing system is the planned Kazakh Latin script closest to? | KZ-14      | Embedding, because the query paraphrases the chunk instead of using exactly the same wording |

In [12]:
own_queries = [
    {
        "id": "Q-09",
        "text": "Who was Nur-Sultan named after in 2019?",
        "relevant": ["KZ-05"]
    },
    {
        "id": "Q-10",
        "text": "Which language's writing system is the planned Kazakh Latin script closest to?",
        "relevant": ["KZ-14"]
    }
]

all_queries = queries["queries"] + own_queries

print("Fixed queries:", len(queries["queries"]))
print("Own queries:", len(own_queries))
print("Total queries:", len(all_queries))

Fixed queries: 8
Own queries: 2
Total queries: 10


## Results for all ten or more queries: top 3 from each retriever

In [13]:
def print_results(results):
    for rank, result in enumerate(results, start=1):
        first_line = result["text"].split("\n")[0]

        print(
            f"{rank}. "
            f"{result['id']} | "
            f"score={result['score']:.4f} | "
            f"{first_line}"
        )

In [14]:
def hit_at_3(results, relevant_ids):
    retrieved_ids = [
        result["id"]
        for result in results
    ]

    return int(
        any(
            chunk_id in relevant_ids
            for chunk_id in retrieved_ids
        )
    )

In [17]:
for item in all_queries:
    query_id = item["id"]
    query_text = item["text"]
    relevant_ids = item["relevant"]

    bm25_results = bm25_retrieve(query_text)
    embedding_results = embedding_retrieve(query_text)

    print("=" * 70)
    print(query_id)
    print("Query:", query_text)
    print("Relevant:", relevant_ids)

    print("\nBM25:")
    print_results(bm25_results)
    print("BM25 hit@3:", hit_at_3(bm25_results, relevant_ids))

    print("\nEmbedding:")
    print_results(embedding_results)
    print("Embedding hit@3:", hit_at_3(embedding_results, relevant_ids))

Q-01
Query: What is form GR-305 used for?
Relevant: ['GO-08', 'GO-12', 'GO-23']

BM25:
1. GO-08 | score=15.8255 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
2. GO-12 | score=8.1953 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.
3. GO-06 | score=8.0700 | Form GR-104 is the application form for the need-based study grant. It is the only way 

## B3 — comparison

| Query           | BM25 top 3 (ids)    | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|-----------------|---------------------|:----------:|-----------------------|:---------------:|
| Q-01            | GO-08, GO-12, GO-06 |     1      | GO-08, GO-12, GO-23   |        1        |
| Q-02            | KZ-17, GO-20, KZ-25 |     0      | GO-01, GO-10, GO-20   |        1        |
| Q-03            | GO-14, GO-23, GO-22 |     1      | GO-14, GO-09, GO-11   |        1        |
| Q-04            | GO-05, G0-17, GO-19 |     0      | GO-02, GO-05, GO-17   |        1        |
| Q-05            | KZ-05, KZ-06, KZ-11 |     1      | KZ-05, KZ-06, KZ-03   |        1        |
| Q-06            | KZ-37, KZ-38, KZ-03 |     1      | KZ-37, KZ-38, KZ-10   |        1        |
| Q-07            | GO-21, GO-16, GO-01 |     0      | GO-15, GO-18, GO-01   |        1        |
| Q-08            | GO-17, GO-05, GO-04 |     1      | GO-17, GO-18, GO-15   |        1        |
| Q-09            | KZ-05, KZ-06, GO-18 |     1      | KZ-05, KZ-14, KZ-04   |        1        |
| Q-10            | KZ-14, KZ-43, KZ-13 |     1      | KZ-14, KZ-43, KZ-12   |        1        |
| **hit@3 total** | 7                   |    /10     | 10                    |       /10       |

**Both retrievers agree:** which query, and why was it easy?

> Q-06 is a example where both retrievers agree. Both return Kz-37 and KZ-38 in the top result. The query is easy and have the same words how in document

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> In my result Embedding show the best result and find the documentary correctly. I think it depends on language of query, because BM25 the most difficult find the correct words.

**Embeddings did better:** which query, and what could BM25 not match?

> Q-07 is the best example, because chunk GO-15 was written in Russian language. BM25 couldn't match the English query with the Russian wording, but Embedding understood the meaning across languages

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [18]:
import json
import os
import tiktoken

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

print("OpenAI key found:", openai_api_key is not None)

client = OpenAI(api_key=openai_api_key)

with open("data/qa.json", "r", encoding="utf-8") as file:
    qa_data = json.load(file)

print(type(qa_data))
print(qa_data.keys() if isinstance(qa_data, dict) else "")

OpenAI key found: True
<class 'dict'>
dict_keys(['note', 'questions'])


In [19]:
questions = qa_data["questions"]

print("Number of questions:", len(questions))
print("First question:")
print(questions[0])

Number of questions: 6
First question:
{'id': 'C-01', 'question': 'How much does a student in income band 2 receive, and when is it paid?', 'reference': '150,000 tenge for the academic year, in two equal instalments: October and February.', 'relevant': ['GO-04', 'GO-15']}


## C1 — the complete prompt, printed before it is sent

In [20]:
SYSTEM_PROMPT = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you don't know.
Cite the context blocks you used, like [1] or [2][3]."""

In [21]:
def build_context(results):
    blocks = []

    for number, result in enumerate(results, start=1):
        block = f"[{number}]\n{result['text']}"
        blocks.append(block)

    return "\n\n".join(blocks)

In [22]:
c1_question = questions[0]

question_text = c1_question["question"]

print("Question:", question_text)

Question: How much does a student in income band 2 receive, and when is it paid?


In [23]:
c1_results = embedding_retrieve(
    question_text,
    top_k=3
)

for rank, result in enumerate(c1_results, start=1):
    print(
        rank,
        result["id"],
        f"score={result['score']:.4f}"
    )

1 GO-04 score=0.5845
2 GO-03 score=0.4842
3 GO-15 score=0.4506


In [24]:
c1_context = build_context(c1_results)

c1_user_prompt = f"""CONTEXT

{c1_context}

QUESTION

{question_text}"""

In [25]:
print("SYSTEM MESSAGE")
print("=" * 70)
print(SYSTEM_PROMPT)

print("\nUSER MESSAGE")
print("=" * 70)
print(c1_user_prompt)

SYSTEM MESSAGE
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you don't know.
Cite the context blocks you used, like [1] or [2][3].

USER MESSAGE
CONTEXT

[1]
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2]
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, whic

In [26]:
encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    return len(encoding.encode(text))

system_tokens = count_tokens(SYSTEM_PROMPT)
question_tokens = count_tokens(question_text)
context_tokens = count_tokens(c1_context)

manual_total = (
    system_tokens
    + question_tokens
    + context_tokens
)

print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Total:", manual_total)

System instructions: 43
Question: 18
Retrieved context: 288
Total: 349


In [27]:
chunk_token_counts = []

for i, result in enumerate(c1_results, start=1):
    tokens = count_tokens(result["text"])
    chunk_token_counts.append(tokens)

    print(
        f"Chunk {i} ({result['id']}):",
        tokens,
        "tokens"
    )

print()
print("Number of retrieved chunks:", len(c1_results))
print("Tokens per chunk:", chunk_token_counts)
print("Total context tokens:", context_tokens)
print("Total context characters:", len(c1_context))

Chunk 1 (GO-04): 90 tokens
Chunk 2 (GO-03): 104 tokens
Chunk 3 (GO-15): 85 tokens

Number of retrieved chunks: 3
Tokens per chunk: [90, 104, 85]
Total context tokens: 288
Total context characters: 1145


In [28]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": c1_user_prompt
        }
    ]
)

c1_answer = response.choices[0].message.content

print("MODEL ANSWER:")
print(c1_answer)

MODEL ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]


In [29]:
api_prompt_tokens = response.usage.prompt_tokens

print("usage.prompt_tokens:", api_prompt_tokens)

usage.prompt_tokens: 364


| Part of the input                           | Tokens (tiktoken `o200k_base`) |
|---------------------------------------------|-------------------------------:|
| System instructions                         |                             43 |
| Question                                    |                             18 |
| Retrieved context                           |                            288 |
| **Total input**                             |                            349 |
| **`usage.prompt_tokens` from the response** |                            364 |

- Number of retrieved chunks: 3
- Tokens per chunk: [90, 104, 85]
- Total context size (tokens / characters): 288 tokens/1145 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>Tiktoken calculation counts only the text that I created, the system instructions, question, and others. The API also adds formatting information for chat messages, such as message roles and internal message structure

**What exactly does retrieval add to the model's input?**

>Retrieval adds the text of the selected chunks to the model's prompt. In this example, the retriever selected 3 chunks and my code inserted them as context blocks [1], [2], and [3].

## C2 — three systems, six questions

In [30]:
def llm_only(question):
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content

In [32]:
def rag_answer(question, retriever):
    results = retriever(
        question,
        top_k=3
    )

    context = build_context(results)

    user_prompt = f"""CONTEXT

{context}

QUESTION

{question}"""

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, results

In [33]:
c2_results = {}

for item in questions:
    question_id = item["id"]
    question_text = item["question"]
    reference = item["reference"]

    print("=" * 80)
    print(question_id)
    print("QUESTION:")
    print(question_text)

    print("\nREFERENCE:")
    print(reference)

    llm_answer = llm_only(question_text)

    bm25_answer, bm25_chunks = rag_answer(
        question_text,
        bm25_retrieve
    )

    embedding_answer, embedding_chunks = rag_answer(
        question_text,
        embedding_retrieve
    )

    c2_results[question_id] = {
        "question": question_text,
        "reference": reference,

        "llm_only": llm_answer,

        "bm25_answer": bm25_answer,
        "bm25_chunks": bm25_chunks,

        "embedding_answer": embedding_answer,
        "embedding_chunks": embedding_chunks
    }

    print("\nLLM ONLY:")
    print(llm_answer)

    print("\nBM25 RAG:")
    print(bm25_answer)

    print("\nEMBEDDING RAG:")
    print(embedding_answer)

    print()

C-01
QUESTION:
How much does a student in income band 2 receive, and when is it paid?

REFERENCE:
150,000 tenge for the academic year, in two equal instalments: October and February.

LLM ONLY:
Which student support scheme or country do you mean? “Income band 2” is used by several programs, and the amount and payment schedule vary. Please share the scheme name, institution, or a link/document.

BM25 RAG:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

EMBEDDING RAG:
A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

C-02
QUESTION:
My grant application was refused. How long do I have to appeal, and which form do I use?

REFERENCE:
14 calendar days from the date on the decision letter, using form GR-305.

LLM ONLY:
The appeal deadline and form 

In [34]:
for question_id, result in c2_results.items():
    print("=" * 80)
    print(question_id)

    print("\nREFERENCE:")
    print(result["reference"])

    print("\nLLM ONLY:")
    print(result["llm_only"])

    print("\nBM25 RAG:")
    print(result["bm25_answer"])

    print("\nEMBEDDING RAG:")
    print(result["embedding_answer"])

C-01

REFERENCE:
150,000 tenge for the academic year, in two equal instalments: October and February.

LLM ONLY:
Which student support scheme or country do you mean? “Income band 2” is used by several programs, and the amount and payment schedule vary. Please share the scheme name, institution, or a link/document.

BM25 RAG:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

EMBEDDING RAG:
A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]
C-02

REFERENCE:
14 calendar days from the date on the decision letter, using form GR-305.

LLM ONLY:
The appeal deadline and form depend on **which grant, agency, and country** you’re dealing with. They should be stated in the refusal/decision letter—often under “appeal,” “review,” or “reconsideration.”

Pleas

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question                    | LLM only                 | BM25 RAG              | Embedding RAG          |
|-----------------------------|--------------------------|-----------------------|------------------------|
| C-01                        | ✗ asks for clarification | ✓ 150.000 tg, Oct/Feb | ✓  150.000 tg, Oct/Feb |
| C-02                        | ✗ doesn't give deadline  | ✓  14 days, GR-305    | ✓ 14 days, GR-305      |
| C-03                        | ✗   doesn't give GPA     | ✓  GPA 2.67/4.00      | ✓  GPA 2.67/4.00       |
| C-04                        | ✓   2017, 2025, 2031     | ✓ 2017, 2025, 2031    | ✓  2017, 2031          |
| C-05                        | ✗ name change            | ✗ name change         | ✗ name change          |
| C-06                        | ✗ US info                | ✓ no amount given     | ✓ no amount given      |
| **agrees with `reference`** | 1/6                      | 5/6                   | 5/6                    |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>GO-16 score=15.7985, GO-02 score=14.0587, GO-06 score=13.8516

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>GO-02 score=0.6286, GO-20 score=0.5691, GO-05 score=0.5451

**3. What was sent to the model** (the context blocks):

>[1]
>The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

>[2]
>An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

>[3]
>Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Retrieval made the answer more accurate and grounded. The LLM-only system couldn't give the GPA, while both RAG systems found GO-02 and correctly answered 2.67.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>Nothing important was missed. Both retrievers returned GO-02. Both models used this information correctly, so there was no retrieval or generation failure.

In [35]:
def show_c3_data(question_id):
    result = c2_results[question_id]

    question = result["question"]

    print("=" * 80)
    print(question_id)
    print("QUESTION:")
    print(question)

    print("\nREFERENCE:")
    print(result["reference"])

    print("\n--- BM25 TOP 3 ---")

    for rank, chunk in enumerate(
        result["bm25_chunks"],
        start=1
    ):
        print(
            rank,
            chunk["id"],
            f"score={chunk['score']:.4f}"
        )
        print(chunk["text"])
        print()

    print("\n--- EMBEDDING TOP 3 ---")

    for rank, chunk in enumerate(
        result["embedding_chunks"],
        start=1
    ):
        print(
            rank,
            chunk["id"],
            f"score={chunk['score']:.4f}"
        )
        print(chunk["text"])
        print()

    print("\n--- BM25 CONTEXT SENT TO MODEL ---")
    print(
        build_context(
            result["bm25_chunks"]
        )
    )

    print("\n--- EMBEDDING CONTEXT SENT TO MODEL ---")
    print(
        build_context(
            result["embedding_chunks"]
        )
    )

    print("\n--- ANSWERS ---")

    print("\nLLM ONLY:")
    print(result["llm_only"])

    print("\nBM25 RAG:")
    print(result["bm25_answer"])

    print("\nEMBEDDING RAG:")
    print(result["embedding_answer"])

show_c3_data("C-03")

C-03
QUESTION:
What is the minimum GPA needed to qualify for the need-based grant?

REFERENCE:
A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

--- BM25 TOP 3 ---
1 GO-16 score=15.7985
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

2 GO-02 score=14.0587
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office re

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>KZ-44 score=17.5729, KZ-11 score=13.7604, KZ-07 score=12.2596

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>KZ-44 score=0.6603, KZ-05 score=0.2715, KZ-14 score=0.2508

**3. What was sent to the model** (the context blocks):

>--- BM25 CONTEXT SENT TO MODEL ---
[1]
Narxoz University ( nar-HOHZ; Kazakh: Нархоз университеті / Narhoz universitetı [nɑrˈχoz ʊnʲɪvʲɪrsʲɪˌtʲeˈtɘ]) is a university in Almaty, Kazakhstan. It offers courses in economics, business, finance, and law. The university was founded in 1963 as Almaty Institute of National Economy and received its current name in 2016. It consists of three schools, five representative offices in the regions of Kazakhstan, and an economic college. The university's courses have been accredited by the Central and East European Management Development Association and Association of Chartered Certified Accountants.

>[2]
In 1955, the Soviet Ministry of Defense issued a decree and founded the Baikonur Cosmodrome. It was originally built as the chief base of operations for the Soviet space program. The Cosmodrome served as the launching point for Sputnik 1 and Vostok 1. The launchpad used for both missions was renamed "Gagarin's Start" in honour of Soviet cosmonaut Yuri Gagarin, who piloted Vostok 1 and became the first human in outer space. As of 2015 under Russian management, Baikonur is a spaceport, with numerous commercial, military, and scientific missions being launched annually.

>[3]
It has been a UNESCO Creative City of Music since 2017, and classified as a Gamma+ level global city by the Globalization and World Cities Research Network. Landmarks and attractions in Almaty include the Central State Museum of Kazakhstan; Ascension Cathedral; Green Bazaar; Arbat, a pedestrian area; and Kök Töbe, a hilltop recreation area with views of the city and the surrounding mountains. The city is also home to educational institutions such as Al-Farabi Kazakh National University, the Kazakh-British Technical University, the International Information Technology University, and Narxoz University.

>--- EMBEDDING CONTEXT SENT TO MODEL ---
[1]
Narxoz University ( nar-HOHZ; Kazakh: Нархоз университеті / Narhoz universitetı [nɑrˈχoz ʊnʲɪvʲɪrsʲɪˌtʲeˈtɘ]) is a university in Almaty, Kazakhstan. It offers courses in economics, business, finance, and law. The university was founded in 1963 as Almaty Institute of National Economy and received its current name in 2016. It consists of three schools, five representative offices in the regions of Kazakhstan, and an economic college. The university's courses have been accredited by the Central and East European Management Development Association and Association of Chartered Certified Accountants.

>[2]
Astana is the capital city of Kazakhstan. With a population of 1,7 million within the city limits, it is the second-largest in the country after Almaty, the national capital until 1997. The city lies on the banks of the Ishim river in the north of Kazakhstan and is located within the Akmola Region but is self-administering. Initially founded as Aqmoly in 1830, the city was later renamed Akmolinsk, Tselinograd, and Aqmola before adopting the name Astana in 1998, or "capital" in Kazakh. In 2019, the city adopted the name Nur-Sultan in honor of former president Nursultan Nazarbayev, but it reverted to the name Astana in 2022. Astana's history includes growth, rapidly growing after becoming the capital. The decision to turn Astana into a modern and planned city was guided by Kisho Kurokawa. It has become an important center for culture, education, and commerce in Kazakhstan and Central Asia.

>[3]
In October 2017, the then president, Nursultan Nazarbayev, decreed that the writing system would change from Cyrillic to Latin script by 2025. The proposed Latin alphabet has been revised several times and, as of January 2021, is close to the inventory of the Turkish alphabet, though it lacks the letters C and Ç and has four additional letters: Ä, Ñ, Q, and Ū (though other letters, such as Y, have different values in the two languages). It is scheduled to be phased in from 2023 to 2031. Over one million Kazakh speakers in Xinjiang use a modified version of the Perso-Arabic script for writing.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Retrieval made the answer more grounded, because both RAG systems correctly found that the university was founded in 1963. However, the answer was incomplete because it didn't mention the 2016 name change

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>The answer missed that the university took its current name in 2016. This was a generation problem because this fact was present in the retrieved context, but the model didn't include it in the answer.

In [36]:
show_c3_data("C-05")

C-05
QUESTION:
When was Narxoz University founded, and what was it called at the start?

REFERENCE:
In 1963, as the Almaty Institute of National Economy; it took its current name in 2016.

--- BM25 TOP 3 ---
1 KZ-44 score=17.5729
Narxoz University ( nar-HOHZ; Kazakh: Нархоз университеті / Narhoz universitetı [nɑrˈχoz ʊnʲɪvʲɪrsʲɪˌtʲeˈtɘ]) is a university in Almaty, Kazakhstan. It offers courses in economics, business, finance, and law. The university was founded in 1963 as Almaty Institute of National Economy and received its current name in 2016. It consists of three schools, five representative offices in the regions of Kazakhstan, and an economic college. The university's courses have been accredited by the Central and East European Management Development Association and Association of Chartered Certified Accountants.

2 KZ-11 score=13.7604
In 1955, the Soviet Ministry of Defense issued a decree and founded the Baikonur Cosmodrome. It was originally built as the chief base of operati

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>GO-16 score=16.9958, GO-01 score=14.1725, GO-20 score=13.7202

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>GO-20 score=0.6664, GO-16 score=0.6161, GO-01 score=0.5539

**3. What was sent to the model** (the context blocks):

>--- BM25 CONTEXT SENT TO MODEL ---
[1]
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

>[2]
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.

>[3]
The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.

>--- EMBEDDING CONTEXT SENT TO MODEL ---
[1]
The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.

>[2]
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

>[3]
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.


**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Retrieval made the answer more grounded. The LLM-only system invented information about US grants, while both RAG systems used the handbook and correctly said that master's students have a separate scheme.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>No grant amount was available in the corpus. The retrieved context says that master's students. Therefore, the RAG systems correctly avoided inventing an amount. There was no retrieval or generation failure.

In [37]:
show_c3_data("C-06")

C-06
QUESTION:
I am a master's student. How much need-based grant can I get?

REFERENCE:
The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

--- BM25 TOP 3 ---
1 GO-16 score=16.9958
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

2 GO-01 score=14.1725
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as exp

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 mainly searches by matching words. Embedding retrieval searches by similarity in meaning.

**2. When can BM25 perform better than embeddings?**

>BM25 can perform better when the query contains exact names, codes, numbers, or words that also appear in the document.

**3. When can embeddings perform better than BM25?**

>Embeddings can perform better with paraphrases, synonyms, or different languages because they compare meaning instead of only exact words.

**4. Why is cosine similarity used with embeddings?**

>Cosine similarity compares the direction of two vectors rather than their length. Similar directions usually mean similar meanings.

**5. What information is actually sent to an LLM in a RAG system?**

>The LLM receives the question, system instructions, and the text of the chunks selected by the retriever. It doesn't receive the whole corpus.

**6. Does the LLM search the original documents itself?**

>No, the retriever searches the documents first, and the selected chunks are then added to the LLM prompt.

**7. What happens if retrieval returns an irrelevant chunk?**

>The model may give a wrong or less accurate answer because it receives irrelevant information as context.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>A larger Top-K can help because it gives the model more chances to receive the correct information. However, too many chunks can add irrelevant or conflicting information and make the answer worse.

>For C-03,  all produced the correct answer, 2.67. K=1 already contained the correct chunk, while larger K values added more context, including less relevant information. This shows that increasing K doesn't always improve the final answer.

In [38]:
def rag_answer_with_k(question, retriever, top_k):
    results = retriever(
        question,
        top_k=top_k
    )

    context = build_context(results)

    user_prompt = f"""CONTEXT

{context}

QUESTION

{question}"""

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, results

In [39]:
c03 = next(
    item for item in questions
    if item["id"] == "C-03"
)

c03_question = c03["question"]

In [40]:
for k in [1, 3, 8]:
    answer, chunks = rag_answer_with_k(
        c03_question,
        embedding_retrieve,
        top_k=k
    )

    print("=" * 70)
    print("Top-K =", k)
    print("Retrieved:", [chunk["id"] for chunk in chunks])
    print("Answer:", answer)

Top-K = 1
Retrieved: ['GO-02']
Answer: The minimum GPA is **2.67 on a 4.0 scale**. [1]
Top-K = 3
Retrieved: ['GO-02', 'GO-20', 'GO-05']
Answer: The minimum GPA is **2.67 on a 4.0 scale**. [1]
Top-K = 8
Retrieved: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
Answer: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]


**9. Is a vector database required to build a RAG system?**

>No, we can use other possible options: library in memory, any database (PostgreSQL or different), BM25, or array list

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>Embedding retrieval worked better overall. It achieved hit@3 on 10/10 queries, while BM25 achieved 7/10. In C2, both BM25 RAG and Embedding RAG agreed with the reference on 5/6 questions.

---
# Conclusion

>Embedding retrieval performed better on this corpus, especially for paraphrases and multilingual queries